# CellLineSelector — Enriched Harmonisation (newer sources)

`00_harmonisation.ipynb` harmonises the 15 original cleaned tables (DepMap/CCLE,
Cellosaurus, GEO, HPA) onto `model_id` and writes `celllineselector.db`. This
notebook is the **additive second wave**: it applies the same method to five
sources that arrived later and were never harmonised.

| source | grain | what it adds |
|---|---|---|
| `gdsc_models` | one row per Sanger model | the **SIDM ↔ ACH ↔ CVCL ↔ CCLE ↔ COSMIC** bridge, plus tissue / cancer type / ploidy / MSI |
| `procan_proteomics` | one row per line | DIA-MS proteomics, 948 lines × 8,453 proteins — 2.5× the CCLE/Gygi line count |
| `cosmic_cna` | one row per (line, gene) | gene-level total copy number |
| `hgnc` | one row per gene | the **authoritative** protein-coding reference |
| `cosmic_cgc` | one row per gene | Cancer Gene Census role (oncogene / TSG), tier |

## Method — unchanged from Stage 0

Same four principles, same helpers, same flag-explode-unwrap sequence:

- **Flag, don't drop** — ambiguous identities are retained and marked
- **Credit both** — a row mapping to 2 candidate ACHs is duplicated, one row each
- **Grain before joining** — event tables aggregated before any join
- **Identity hubs hold lists** — one row per entity, alternative ids as list columns

## What is new here

Stage 0 resolved each table through **one** identifier axis (a table was CVCL-keyed,
or profile-keyed, or GSM-keyed). The newer sources each carry several usable ids at
once, so this notebook adds a **priority-ordered multi-path resolver**: try the
strongest axis first, fall back down the chain, and record which axis actually
matched in a `matched_via` column. A match that came from a name is not as
trustworthy as one that came from an accession, and the column keeps that
difference inspectable rather than averaging it away.

## Flow

```
load -> structural normalisation -> extend identity hub (Sanger axis)
     -> multi-path resolve -> extend coverage -> enrich gene dimension
     -> persist -> verify
```

Reads and extends `outputs/celllineselector.db`. Never modifies a Stage 0 table —
every output is a new `*_enriched` table or a new source table.

## 0. Configuration

Every constant used downstream is declared here, so nothing is redefined mid-notebook.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import duckdb

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

# repo root, from src/pipeline
ROOT    = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
DATA    = os.path.join(ROOT, "data")
OUT_DIR = os.path.abspath(os.path.join(os.getcwd(), "outputs"))
DB_PATH = os.path.join(OUT_DIR, "celllineselector.db")

SRC_FILES = {
    "gdsc_models": os.path.join(DATA, "GDSC", "model_list_20260709.csv"),
    "procan":      os.path.join(DATA, "proteomics_procan", "Protein_matrix_averaged_20250211.tsv"),
    "cosmic_cna":  os.path.join(DATA, "COSMIC", "CellLinesProject_CompleteCNA_v104_GRCh37.tsv"),
    "cosmic_cgc":  os.path.join(DATA, "COSMIC", "Cosmic_CancerGeneCensus_v104_GRCh37.tsv"),
    "hgnc":        os.path.join(DATA, "gene_with_protein_product.txt"),
}

# Containers that model_id may arrive wrapped in, before flattening
LIST_LIKE = (list, tuple, set, frozenset, np.ndarray, pd.Series, pd.Index)

# New MEASURED layers this notebook adds. gdsc_models is annotation, not a
# measurement, so it is kept out of the modality count for the same reason
# sample_info is: it would inflate every line it touches.
NEW_MODALITY_LAYERS   = ["procan_proteomics", "cosmic_cna"]
NEW_ANNOTATION_LAYERS = ["gdsc_models"]

# Stage 0 layers, repeated so the enriched coverage matrix is a superset
MODALITY_LAYERS   = ["depmap_expr", "geo_expr", "hpa_rna", "mutations", "fusions",
                     "proteomics", "metabolomics", "mirna", "signatures"]
ANNOTATION_LAYERS = ["sample_info", "cellosaurus", "hpa_desc", "depmap_profiles", "geo_info"]

ALL_MODALITY_LAYERS = MODALITY_LAYERS + NEW_MODALITY_LAYERS
ALL_LAYERS          = ALL_MODALITY_LAYERS + ANNOTATION_LAYERS + NEW_ANNOTATION_LAYERS

ENSG_PATTERN = r'^ensg\d+'

# COSMIC's total-copy-number calls. Thresholds match build_cna_layer.py so the
# two never disagree about what counts as an amplification.
AMP_THRESHOLD = 2.5
DEL_THRESHOLD = 1.5

for k, p in SRC_FILES.items():
    print(f"{k:14s} exists={os.path.isfile(p)}  {p}")
print(f"\nDB_PATH: {DB_PATH}  exists={os.path.isfile(DB_PATH)}")

## 1. Shared helpers

Redefined here rather than imported, so this notebook runs standalone against an
existing database. The first six are byte-identical to Stage 0; `resolve_model_id`
and `attach_model_id` are new.

In [ ]:
def lower_all(df):
    """Lowercase and strip column headers and all string values."""
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower()
    for c in df.select_dtypes(include=["object", "string"]).columns:
        df[c] = df[c].astype("string").str.strip().str.lower()
    return df


def normalise_accession(s, drop_prefix=None):
    """Normalise any CVCL-style identifier to bare 'cvcl_xxxx'."""
    s = s.astype("string").str.strip().str.lower()
    if drop_prefix:
        s = s.str.replace(rf"^{drop_prefix}[:_\-\s]*", "", regex=True)
    return (s.str.replace(r"^cvcl[:_\-\s]*", "cvcl_", regex=True)
             .str.replace(r"[^a-z0-9_]+$", "", regex=True))


def explode_ids(series):
    """Flatten scalars / list-likes / nested list-likes into one flat Series."""
    s = series.dropna()
    for _ in range(10):
        if not s.map(lambda x: isinstance(x, LIST_LIKE)).any():
            break
        s = s.explode().dropna()
    return s


def build_lookup(roster, list_col, key_col="model_id"):
    """element-of-`list_col` -> sorted list of model_ids. Always returns a LIST."""
    if list_col not in roster.columns:
        print(f"  [skip] lookup on '{list_col}': column not present")
        return pd.Series(dtype="object")
    flat = roster[[key_col, list_col]].explode(list_col).dropna(subset=[list_col])
    return flat.groupby(list_col)[key_col].agg(lambda s: sorted(set(s)))


def union_lists(values):
    """Union of all list values in a Series, ignoring NaN."""
    out = set()
    for v in values.dropna():
        out.update(v)
    return sorted(out)


def collect(df, val, name, key="model_id"):
    """model_id -> sorted set of `val`, as a named Series, for the identity hub."""
    if key not in df.columns or val not in df.columns:
        missing = key if key not in df.columns else val
        print(f"  [skip] {name}: column '{missing}' not present")
        return None
    d = df[[key, val]].dropna()
    if d[key].map(lambda x: isinstance(x, LIST_LIKE)).any():
        d = d.explode(key).dropna(subset=[key])
    return d.groupby(key)[val].agg(lambda s: sorted(set(s.dropna()))).rename(name)


def explode_model_id(df, name=""):
    """Flag ambiguity, then split list-valued model_id into one row per candidate."""
    if "model_id" not in df.columns:
        return df
    out = df.copy()
    out["n_model_id"] = out["model_id"].map(
        lambda x: len(x) if isinstance(x, LIST_LIKE) else (0 if pd.isna(x) else 1))
    out["is_ambiguous"] = out["n_model_id"] > 1
    before = len(out)
    out = out.explode("model_id").reset_index(drop=True)
    print(f"  {name:18s} {before:>9,} -> {len(out):>9,} rows  "
          f"(+{len(out) - before}, ambiguous {int(out['is_ambiguous'].sum())}, "
          f"unmatched {int((out['n_model_id'] == 0).sum())})")
    return out


def unwrap_model_id(df, name=""):
    """Convert model_id from list type to plain string, AFTER exploding."""
    if df is None or "model_id" not in df.columns:
        return df
    out = df.copy()

    def unwrap(x):
        if isinstance(x, LIST_LIKE):
            if len(x) == 0:
                return np.nan
            if len(x) == 1:
                return x[0]
            return x
        return x

    out["model_id"] = out["model_id"].map(unwrap)
    n_still = int(out["model_id"].map(lambda x: isinstance(x, LIST_LIKE)).sum())
    out["model_id"] = out["model_id"].astype("string")
    flag = "  <-- explode this table first!" if n_still else ""
    print(f"  {name:18s} dtype {out['model_id'].dtype}, still-list rows: {n_still}{flag}")
    return out


def clean_ensg(s):
    """Lowercase, trim, and strip the Ensembl version suffix (.12)."""
    return (s.astype(str).str.strip().str.lower()
             .str.replace(r"\.\d+$", "", regex=True))


def report_cardinality(df, left, right, label=""):
    """Print the cardinality of left <-> right in both directions."""
    l2r = df.dropna(subset=[left]).groupby(left)[right].nunique()
    r2l = df.dropna(subset=[right]).groupby(right)[left].nunique()
    print(f"--- {label or f'{left} <-> {right}'} ---")
    print(f"{left:>28s} with >1 {right}: {(l2r > 1).sum():5d} / {len(l2r)}  (max {l2r.max()})")
    print(f"{right:>28s} with >1 {left}: {(r2l > 1).sum():5d} / {len(r2l)}  (max {r2l.max()})")
    print(f"strictly 1:1: {(l2r.max() == 1) and (r2l.max() == 1)}\n")

### 1.1 The multi-path resolver

**New in this notebook.** Each newer source carries several usable identifiers at
once, so resolution is a priority chain rather than a single lookup.

The order is by how much the identifier can be trusted:

1. **ACH directly** — the key itself, no inference
2. **CVCL / RRID** — a stable curated accession
3. **CCLE name** — a structured `{NAME}_{TISSUE}` id, unique by construction
4. **SIDM** — Sanger's model id, resolved through the hub axis built in Section 4
5. **plain name** — last resort; short codes collide, so this is the weakest link

`matched_via` records which rung actually fired. A downstream consumer that only
trusts accession-level identity can filter on it; one that wants maximum reach can
ignore it. Either way the distinction is preserved rather than being flattened into
an undifferentiated "matched".

The return is always a **list**, so a key that resolves to two ACHs stays ambiguous
until `explode_model_id` credits both — exactly as in Stage 0.

In [ ]:
def resolve_model_id(df, chain, name=""):
    """
    Resolve model_id through a priority-ordered chain of identifier axes.

    chain: list of (label, source_column, lookup_series). Each lookup maps a key
    to a LIST of candidate ACHs. The first rung that produces a non-empty list
    wins for that row; `matched_via` records which one it was.

    Rows resolved by no rung get model_id = NaN and matched_via = 'unmatched'.
    They are never dropped here — Section 6 counts them.
    """
    out = df.copy()
    resolved   = pd.Series([None] * len(out), index=out.index, dtype="object")
    matched_by = pd.Series("unmatched", index=out.index, dtype="object")

    for label, col, lookup in chain:
        if col not in out.columns or lookup is None or len(lookup) == 0:
            print(f"  {name:18s} [skip] {label:12s} (column '{col}' or lookup missing)")
            continue
        todo = resolved.isna()
        if not todo.any():
            break
        hit = out.loc[todo, col].map(lookup)
        got = hit.map(lambda x: isinstance(x, LIST_LIKE) and len(x) > 0)
        resolved.loc[todo[todo].index[got.values]]   = hit[got.values]
        matched_by.loc[todo[todo].index[got.values]] = label
        print(f"  {name:18s} {label:12s} +{int(got.sum()):>7,} rows "
              f"({int(resolved.notna().sum()):>7,}/{len(out):,} resolved)")

    out["model_id"]   = resolved
    out["matched_via"] = matched_by
    n = int(resolved.notna().sum())
    print(f"  {name:18s} TOTAL        {n:>7,}/{len(out):,} ({n / max(len(out), 1) * 100:.1f}%)")
    return out


def attach_model_id(df, chain, name=""):
    """resolve -> flag+explode ambiguity -> unwrap to plain string. One call."""
    out = resolve_model_id(df, chain, name)
    out = explode_model_id(out, name)
    out = unwrap_model_id(out, name)
    return out

## 2. Load

Stage 0's database is opened first — the identity hub and gene dimension it built
are the reference every newer source resolves against.

In [ ]:
if not os.path.isfile(DB_PATH):
    raise FileNotFoundError(
        f"{DB_PATH} not found. Run 00_harmonisation.ipynb first — this notebook "
        "extends the database it builds, it does not create one.")

con = duckdb.connect(DB_PATH)

cell_line_connection = con.execute("SELECT * FROM cell_line_connection").df()
gene                 = con.execute("SELECT * FROM gene").df()
coverage_matrix      = con.execute("SELECT * FROM coverage_matrix").df()
sample_info          = con.execute("SELECT * FROM sample_info").df()

print(f"cell_line_connection : {cell_line_connection.shape}")
print(f"gene                 : {gene.shape}")
print(f"coverage_matrix      : {coverage_matrix.shape}")
print(f"sample_info          : {sample_info.shape}")

### 2.1 The five newer sources

`procan` needs special handling: the file carries **three header rows** before the
data — UniProt accessions, gene symbols, then the two id column names. It is read
in two passes, headers then body, rather than letting pandas guess.

In [ ]:
# --- GDSC / Sanger model list -------------------------------------------
gdsc_models = pd.read_csv(SRC_FILES["gdsc_models"], low_memory=False)
print(f"gdsc_models  {gdsc_models.shape}")

# --- ProCan DIA-MS proteomics -------------------------------------------
# row 0 = uniprot accessions, row 1 = gene symbols, row 2 = id column names,
# rows 3+ = one row per cell line
_hdr = pd.read_csv(SRC_FILES["procan"], sep="\t", nrows=3, header=None, low_memory=False)
procan_uniprots = [str(u).strip().lower() for u in _hdr.iloc[0, 2:].tolist()]
procan_symbols  = [str(s).strip().lower() for s in _hdr.iloc[1, 2:].tolist()]

_body = pd.read_csv(SRC_FILES["procan"], sep="\t", skiprows=3, header=None, low_memory=False)
procan = pd.DataFrame(_body.iloc[:, 2:].to_numpy(dtype="float32"), columns=procan_uniprots)
procan.insert(0, "sanger_model_id", _body.iloc[:, 1].astype("string").str.strip().str.lower())
procan.insert(0, "gdsc_model_name", _body.iloc[:, 0].astype("string").str.strip().str.lower())
print(f"procan       {procan.shape}  ({len(procan_uniprots):,} proteins)")

# --- COSMIC cell lines CNA ----------------------------------------------
cosmic_cna = pd.read_csv(SRC_FILES["cosmic_cna"], sep="\t", usecols=[
    "COSMIC_SAMPLE_ID", "SAMPLE_NAME", "COSMIC_GENE_ID", "GENE_SYMBOL",
    "TOTAL_CN", "MINOR_ALLELE", "MUT_TYPE", "CHROMOSOME"])
print(f"cosmic_cna   {cosmic_cna.shape}")

# --- COSMIC Cancer Gene Census ------------------------------------------
cosmic_cgc = pd.read_csv(SRC_FILES["cosmic_cgc"], sep="\t")
print(f"cosmic_cgc   {cosmic_cgc.shape}")

# --- HGNC protein-coding reference --------------------------------------
hgnc = pd.read_csv(SRC_FILES["hgnc"], sep="\t", low_memory=False, usecols=[
    "hgnc_id", "symbol", "name", "locus_group", "locus_type", "status",
    "alias_symbol", "prev_symbol", "entrez_id", "ensembl_gene_id", "uniprot_ids"])
print(f"hgnc         {hgnc.shape}")

## 3. Structural normalisation

Source-shape work only — naming, formats, id casing. All of it before any identity
logic, exactly as in Stage 0 Section 3.

### 3.1 `gdsc_models` — canonical id casing

Every id column is lowercased to match the rest of the warehouse, and `RRID` is put
through the same `normalise_accession` used on `cellosaurus_accession` and
`sample_info.rrid` in Stage 0. Without that, `CVCL_4897` and `cvcl_4897` are two
different strings and the join silently returns nothing.

Organoids are **flagged, not dropped**. 256 of the 2,266 rows are organoids rather
than cell lines; they are a different experimental object, but removing them here
would make that decision invisible to anyone reading the output.

In [ ]:
gdsc_models = lower_all(gdsc_models)
gdsc_models = gdsc_models.rename(columns={
    "model_id":   "sanger_model_id",   # SIDM — 'model_id' means ACH everywhere else
    "model_name": "gdsc_model_name",
    "broad_id":   "broad_ach",
    "ccle_id":    "ccle_name",
    "rrid":       "rrid",
    "cosmic_id":  "cosmic_id",
})

gdsc_models["rrid"]      = normalise_accession(gdsc_models["rrid"])
gdsc_models["broad_ach"] = gdsc_models["broad_ach"].astype("string").str.strip().str.lower()
gdsc_models["ccle_name"] = gdsc_models["ccle_name"].astype("string").str.strip().str.lower()
gdsc_models["cosmic_id"] = gdsc_models["cosmic_id"].astype("string").str.strip()
gdsc_models["is_organoid"] = gdsc_models["model_type"].eq("organoid")

KEEP = ["sanger_model_id", "gdsc_model_name", "broad_ach", "ccle_name", "rrid",
        "cosmic_id", "synonyms", "tissue", "cancer_type", "cancer_type_detail",
        "tissue_status", "sample_site", "model_type", "is_organoid",
        "growth_properties", "msi_status", "ploidy_wes", "ploidy_wgs",
        "mutational_burden", "gender", "ethnicity", "age_at_sampling", "species"]
gdsc_models = gdsc_models[[c for c in KEEP if c in gdsc_models.columns]]

print(f"gdsc_models: {gdsc_models.shape}")
for c in ["sanger_model_id", "broad_ach", "rrid", "ccle_name", "cosmic_id", "gdsc_model_name"]:
    s = gdsc_models[c]
    print(f"  {c:18s} {s.notna().sum():>5,} non-null | {s.nunique():>5,} distinct "
          f"| dup non-null: {int(s.dropna().duplicated().sum())}")
print(f"\n  organoids flagged: {int(gdsc_models.is_organoid.sum())} "
      f"of {len(gdsc_models)} (kept, not dropped)")

### 3.2 `procan` — protein axis alignment

Stage 0's `proteomics` table uses **lowercased UniProt accessions as column names**
and resolves them to genes at query time through `gene.uniprot_ids`. ProCan is
lowercased the same way, so both proteomics layers sit on one protein axis and no
separate mapping table is needed for the newer one.

The symbol row is kept as a standalone `procan_protein_map` rather than being folded
into the column names — a UniProt accession is stable, a symbol is not.

In [ ]:
procan_protein_map = pd.DataFrame({
    "uniprot_id":  procan_uniprots,
    "gene_symbol": procan_symbols,
}).replace({"gene_symbol": {"nan": pd.NA}})

dupe_uniprots = pd.Series(procan_uniprots).duplicated().sum()
overlap = set(procan_uniprots) & set(
    con.execute("DESCRIBE proteomics").df()["column_name"].str.lower())

print(f"procan proteins        : {len(procan_uniprots):,}  (duplicate accessions: {dupe_uniprots})")
print(f"shared with CCLE/Gygi  : {len(overlap):,}")
print(f"procan-only            : {len(set(procan_uniprots) - overlap):,}")
print(f"symbols resolved       : {procan_protein_map.gene_symbol.notna().sum():,}")
print(f"\nvalue matrix: all-NaN columns = {int(procan[procan_uniprots].isna().all().sum()):,}")

### 3.3 `cosmic_cna` — grain and call derivation

CNA arrives at **(sample, gene) grain** with a raw `TOTAL_CN`. Two things happen here:

- `cna_call` derives amplification / deletion / neutral using the same 2.5 / 1.5
  thresholds as `build_cna_layer.py`, so the two never disagree about what an
  amplification is
- the gene key is a **symbol**, not an ENSG — it is resolved to `gene_id` in Section
  7 against the gene dimension, once HGNC has supplied the alias table that makes
  the symbol match reliable

`MUT_TYPE` (COSMIC's own gain/loss call) is kept alongside `cna_call`, so a
disagreement between COSMIC's threshold and ours is visible rather than overwritten.

In [ ]:
cosmic_cna = lower_all(cosmic_cna)
cosmic_cna = cosmic_cna.rename(columns={
    "sample_name": "cosmic_sample_name", "gene_symbol": "cosmic_gene_symbol"})
cosmic_cna["total_cn"] = pd.to_numeric(cosmic_cna["total_cn"], errors="coerce")

cosmic_cna["cna_call"] = np.select(
    [cosmic_cna["total_cn"] > AMP_THRESHOLD, cosmic_cna["total_cn"] < DEL_THRESHOLD],
    ["amplification", "deletion"], default="neutral")
cosmic_cna.loc[cosmic_cna["total_cn"].isna(), "cna_call"] = "unknown"

print(f"cosmic_cna: {cosmic_cna.shape}")
print(f"  distinct samples : {cosmic_cna.cosmic_sample_name.nunique():,}")
print(f"  distinct genes   : {cosmic_cna.cosmic_gene_symbol.nunique():,}")
print(f"  total_cn range   : {cosmic_cna.total_cn.min():.0f} - {cosmic_cna.total_cn.max():.0f}")
print("\n  cna_call (ours, at 2.5 / 1.5):")
print(cosmic_cna.cna_call.value_counts().to_string())
print("\n  vs COSMIC's own MUT_TYPE:")
print(pd.crosstab(cosmic_cna.cna_call, cosmic_cna.mut_type,
                  margins=True, margins_name="TOTAL").to_string())

### 3.4 `hgnc` and `cosmic_cgc` — gene-axis references

`hgnc` is the file the Stage 0 whitelist was standing in for. Stage 0 §3.6 inferred
the protein-coding gene set by unioning `mutations` and `hpa_rna` and noted openly
that *"a GENCODE biotype table would be the rigorous source; this is a defensible
substitute given what's loaded."* HGNC is that rigorous source, and Section 7
measures how close the inference actually got.

`alias_symbol` and `prev_symbol` are exploded into a long symbol→ENSG table. Symbols
drift between releases: COSMIC calls a gene by a name HGNC has since retired, and
without the alias table that gene silently fails to join.

In [ ]:
hgnc = lower_all(hgnc)
# clean_ensg goes through astype(str), which turns a missing value into the
# literal '<na>'. Anything that isn't a real ENSG is masked back to NA here, or
# that placeholder becomes a 20,000th "gene" in every set operation below.
_ensg = clean_ensg(hgnc["ensembl_gene_id"])
hgnc["ensembl_gene_id"] = _ensg.where(_ensg.str.match(ENSG_PATTERN, na=False), pd.NA)
hgnc = hgnc.rename(columns={"symbol": "hgnc_symbol", "ensembl_gene_id": "gene_id"})

# symbol -> gene_id, across current, alias and previous symbols. '|' separated.
_parts = []
for col, kind in [("hgnc_symbol", "current"), ("alias_symbol", "alias"),
                  ("prev_symbol", "previous")]:
    d = hgnc[["gene_id", col]].dropna()
    d = (d.assign(sym=d[col].astype("string").str.split("|"))
           .explode("sym")[["gene_id", "sym"]])
    d["sym"] = d["sym"].str.strip().str.strip('"').str.lower()
    d = d[d["sym"].notna() & (d["sym"] != "")]
    d["symbol_kind"] = kind
    _parts.append(d)

hgnc_symbols = (pd.concat(_parts, ignore_index=True)
                  .dropna(subset=["gene_id"])
                  .drop_duplicates(["sym", "gene_id", "symbol_kind"]))

print(f"hgnc            : {hgnc.shape}  | with ENSG: {hgnc.gene_id.notna().sum():,}")
print(f"hgnc_symbols    : {len(hgnc_symbols):,} symbol->gene_id pairs")
print(hgnc_symbols.symbol_kind.value_counts().to_string())
amb_sym = hgnc_symbols.groupby("sym")["gene_id"].nunique()
print(f"\nsymbols resolving to >1 gene_id: {(amb_sym > 1).sum():,} of {len(amb_sym):,}")

# --- Cancer Gene Census -------------------------------------------------
cosmic_cgc = lower_all(cosmic_cgc)
cosmic_cgc = cosmic_cgc.rename(columns={"gene_symbol": "cgc_symbol"})


def normalise_role(r):
    """COSMIC writes free text ('oncogene, TSG, fusion'); collapse to one label."""
    if pd.isna(r):
        return "unknown"
    r = str(r).lower()
    tsg, onc = "tsg" in r, "oncogene" in r
    if tsg and onc:
        return "both"
    return "tsg" if tsg else ("oncogene" if onc else "unknown")


cosmic_cgc["gene_role"] = cosmic_cgc["role_in_cancer"].apply(normalise_role)
print(f"\ncosmic_cgc      : {cosmic_cgc.shape}")
print(cosmic_cgc.gene_role.value_counts().to_string())
print("tier:", cosmic_cgc.tier.value_counts().to_dict())

## 4. Extend the identity hub — the Sanger axis

Stage 0's hub holds CVCL, profile, GEO, CCLE and name axes. It has no **SIDM**
(Sanger model id) axis, because none of the 15 original tables carried one in a
usable form — and SIDM is exactly what ProCan is keyed on.

There are two independent routes from SIDM to ACH:

- `gdsc_models`: `sanger_model_id` → `broad_ach`, curated by Sanger
- `sample_info`: `sanger_model_id`, curated by DepMap

They are built separately and then compared. Where the two disagree, that is a
**genuine identity conflict between two curation teams**, not a bug to be resolved
by picking one — so both are kept and `sanger_id_conflict` marks the line, mirroring
how Stage 0 treats `cvcl_accessions` disagreeing with `rrids`.

In [ ]:
# route A — Sanger's own BROAD_ID mapping
sanger_from_gdsc = (gdsc_models.dropna(subset=["broad_ach", "sanger_model_id"])
                      [["broad_ach", "sanger_model_id"]]
                      .rename(columns={"broad_ach": "model_id"}))

# route B — DepMap's sanger_model_id column
sanger_from_depmap = pd.DataFrame(columns=["model_id", "sanger_model_id"])
if "sanger_model_id" in sample_info.columns:
    sanger_from_depmap = (sample_info.dropna(subset=["model_id", "sanger_model_id"])
                            [["model_id", "sanger_model_id"]].copy())
    sanger_from_depmap["sanger_model_id"] = (
        sanger_from_depmap["sanger_model_id"].astype("string").str.strip().str.lower())

print(f"route A (gdsc_models.broad_ach)     : {len(sanger_from_gdsc):,} pairs")
print(f"route B (sample_info.sanger_model_id): {len(sanger_from_depmap):,} pairs")

_a = set(map(tuple, sanger_from_gdsc.values))
_b = set(map(tuple, sanger_from_depmap.values))
print(f"\n  agreed by both routes : {len(_a & _b):,}")
print(f"  route A only          : {len(_a - _b):,}")
print(f"  route B only          : {len(_b - _a):,}")

# same ACH, different SIDM depending on who you ask
both = pd.concat([sanger_from_gdsc, sanger_from_depmap], ignore_index=True).drop_duplicates()
conflict = both.groupby("model_id")["sanger_model_id"].nunique()
conflicted = set(conflict[conflict > 1].index)
print(f"  ACHs where the two routes disagree: {len(conflicted):,}")
if conflicted:
    print(both[both.model_id.isin(sorted(conflicted)[:5])]
            .sort_values("model_id").to_string(index=False))

### 4.1 Attach the new axes to the hub

`sanger_ids`, `cosmic_ids` and `gdsc_model_names` join `rrids`, `profile_ids` and the
rest as **list** columns, so `model_id` stays unique and every join to the hub remains
one-to-one.

The hub base also widens: `gdsc_models` carries ACHs that no Stage 0 table mentions,
the same situation `fusions` created in Stage 0 §4.3. Those lines are added with
`in_roster = False` and `source_wave = 'enriched'`, so a line that only the newer data
knows about is never mistaken for one DepMap has always had.

In [ ]:
enriched_hub = cell_line_connection.copy()
enriched_hub["source_wave"] = "core"

# widen the base with ACHs only the newer sources know about
new_achs = sorted(set(sanger_from_gdsc["model_id"].dropna()) - set(enriched_hub["model_id"]))
if new_achs:
    add = pd.DataFrame({"model_id": new_achs})
    add["in_roster"]   = False
    add["source_wave"] = "enriched"
    enriched_hub = pd.concat([enriched_hub, add], ignore_index=True)
print(f"hub base: {len(cell_line_connection):,} -> {len(enriched_hub):,} ACHs "
      f"(+{len(new_achs)} contributed only by the newer sources)")

# --- new identifier axes -------------------------------------------------
sanger_per_ach = (both.groupby("model_id")["sanger_model_id"]
                    .agg(lambda s: sorted(set(s.dropna()))).rename("sanger_ids"))
enriched_hub = enriched_hub.merge(sanger_per_ach, on="model_id", how="left")
enriched_hub["sanger_id_conflict"] = enriched_hub["model_id"].isin(conflicted)

_gdsc_by_ach = gdsc_models.dropna(subset=["broad_ach"]).rename(columns={"broad_ach": "model_id"})
for _val, _name in [("cosmic_id", "cosmic_ids"), ("gdsc_model_name", "gdsc_model_names")]:
    _s = collect(_gdsc_by_ach, _val, _name)
    if _s is not None:
        enriched_hub = enriched_hub.merge(_s, on="model_id", how="left")

NEW_ID_COLS = ["sanger_ids", "cosmic_ids", "gdsc_model_names"]
for c in NEW_ID_COLS:
    enriched_hub[f"n_{c}"] = enriched_hub[c].map(
        lambda x: len(x) if isinstance(x, LIST_LIKE) else 0)

print(f"\n{'identifier':20s} {'lines with it':>14s}")
for c in NEW_ID_COLS:
    print(f"{c:20s} {(enriched_hub[f'n_{c}'] > 0).sum():>14,}")
print(f"\nACHs with >1 sanger_id (curation conflict): "
      f"{int(enriched_hub['sanger_id_conflict'].sum())}")
print(f"hub: {len(enriched_hub):,} rows, model_id unique: {enriched_hub.model_id.is_unique}")

## 5. Attach `model_id` to every newer table

Every lookup is built from the extended hub, so there is one source of truth per
identifier axis — the same discipline as Stage 0 §5, now over five axes instead of one.

In [ ]:
cvcl_to_models   = build_lookup(enriched_hub, "rrids")
ccle_to_models   = build_lookup(enriched_hub, "ccle_names")
ccle_id_to_models = build_lookup(enriched_hub, "ccle_ids")
sidm_to_models   = build_lookup(enriched_hub, "sanger_ids")
gname_to_models  = build_lookup(enriched_hub, "gdsc_model_names")
name_to_models   = build_lookup(enriched_hub, "stripped_names")
ach_to_models    = pd.Series(enriched_hub["model_id"].map(lambda x: [x]).values,
                             index=enriched_hub["model_id"])

for n, l in [("cvcl", cvcl_to_models), ("ccle_name", ccle_to_models),
             ("ccle_id", ccle_id_to_models), ("sidm", sidm_to_models),
             ("gdsc_name", gname_to_models), ("stripped_name", name_to_models),
             ("ach", ach_to_models)]:
    print(f"  {n:14s} {len(l):>6,} keys")

### 5.1 `gdsc_models`

The full chain, strongest rung first. `broad_ach` should carry most of it; the
remaining rungs measure how much reach the other identifiers add on top.

In [ ]:
print("resolving gdsc_models:")
gdsc_models = attach_model_id(gdsc_models, [
    ("ach",       "broad_ach",        ach_to_models),
    ("cvcl",      "rrid",             cvcl_to_models),
    ("ccle_name", "ccle_name",        ccle_to_models),
    ("name",      "gdsc_model_name",  name_to_models),
], "gdsc_models")

print("\nmatched_via:")
print(gdsc_models.matched_via.value_counts().to_string())

### 5.2 `procan_proteomics`

SIDM-keyed, so the Sanger axis built in Section 4 is what makes this table joinable
at all — without it, ProCan cannot reach ACH by any route except its name column.

In [ ]:
print("resolving procan:")
procan_proteomics = attach_model_id(procan, [
    ("sidm",      "sanger_model_id", sidm_to_models),
    ("gdsc_name", "gdsc_model_name", gname_to_models),
    ("name",      "gdsc_model_name", name_to_models),
], "procan")

print("\nmatched_via:")
print(procan_proteomics.matched_via.value_counts().to_string())

_ccle_n = con.execute("SELECT count(DISTINCT model_id) FROM proteomics").fetchone()[0]
print(f"\nCCLE/Gygi reaches {_ccle_n:,} ACHs; ProCan reaches "
      f"{procan_proteomics.model_id.nunique():,}. Section 6.1 measures the union.")

### 5.3 `cosmic_cna`

COSMIC's `SAMPLE_NAME` is a Sanger model name, so it routes through `gdsc_model_names`
first and only falls back to the DepMap stripped-name axis. This is the same bridge
`build_cna_layer.py` uses, but built from the hub rather than re-derived from the CSV,
so the two cannot drift apart.

`cosmic_cna` is an **event table at (sample, gene) grain** — 168 K rows over ~1,000
samples. Resolution runs on the ~1,000 distinct sample names and is merged back,
rather than pushing every row through a Python lambda: same result, two orders of
magnitude fewer lookups. This is the *grain before joining* principle applied to the
resolution step itself.

In [ ]:
_samples = (cosmic_cna[["cosmic_sample_name"]].drop_duplicates()
              .reset_index(drop=True))
print(f"resolving {len(_samples):,} distinct COSMIC samples "
      f"(not all {len(cosmic_cna):,} rows):")
_samples = attach_model_id(_samples, [
    ("gdsc_name", "cosmic_sample_name", gname_to_models),
    ("name",      "cosmic_sample_name", name_to_models),
    ("ccle_name", "cosmic_sample_name", ccle_to_models),
], "cosmic_samples")

print("\nmatched_via (per sample):")
print(_samples.matched_via.value_counts().to_string())

before = len(cosmic_cna)
cosmic_cna = cosmic_cna.merge(_samples, on="cosmic_sample_name", how="left")
print(f"\nrows: {before:,} -> {len(cosmic_cna):,}  "
      f"(ambiguous samples duplicate rows, per 'credit both')")
print(f"rows with model_id : {cosmic_cna.model_id.notna().sum():,} "
      f"({cosmic_cna.model_id.notna().mean() * 100:.1f}%)")
print(f"distinct ACHs      : {cosmic_cna.model_id.nunique():,}")

## 6. Coverage — what the newer sources actually add

The enriched coverage matrix is a **superset** of Stage 0's: the nine original
modality columns are carried through unchanged and two are appended, so
`n_modalities` is now out of 11. Reading the two side by side shows exactly what the
second wave bought.

In [ ]:
new_layer_ids = {
    "procan_proteomics": set(explode_ids(procan_proteomics["model_id"])),
    "cosmic_cna":        set(explode_ids(cosmic_cna["model_id"])),
    "gdsc_models":       set(explode_ids(gdsc_models["model_id"])),
}

all_ids = set(enriched_hub["model_id"])
coverage_enriched = pd.DataFrame({"model_id": sorted(all_ids)})

# carry Stage 0's columns through
coverage_enriched = coverage_enriched.merge(
    coverage_matrix.drop(columns=["n_modalities", "complete_cycle"], errors="ignore"),
    on="model_id", how="left")
for c in MODALITY_LAYERS + ANNOTATION_LAYERS:
    if c in coverage_enriched.columns:
        # lines added by the enriched wave have no Stage 0 row -> NA, which means
        # "that layer does not hold this line", i.e. False
        coverage_enriched[c] = coverage_enriched[c].astype("boolean").fillna(False).astype(bool)

for name, ids in new_layer_ids.items():
    coverage_enriched[name] = coverage_enriched["model_id"].isin(ids)

coverage_enriched["n_modalities"]   = coverage_enriched[ALL_MODALITY_LAYERS].sum(axis=1)
coverage_enriched["complete_cycle"] = coverage_enriched["n_modalities"] == len(ALL_MODALITY_LAYERS)

print(f"total model_id in enriched hub: {len(all_ids):,}\n")
for name, ids in new_layer_ids.items():
    n = len(ids & all_ids)
    print(f"{name:20s} -> {n:5d} / {len(all_ids)} ({n / len(all_ids) * 100:5.1f}%)")

print(f"\nlines with all {len(ALL_MODALITY_LAYERS)} modalities: "
      f"{int(coverage_enriched['complete_cycle'].sum())}")
print("\ndistribution of n_modalities:")
print(coverage_enriched["n_modalities"].value_counts().sort_index(ascending=False).to_string())

### 6.1 Proteomics reach — the headline gain

`test_run_proteomics_platform_overlap.py` measured the batch effect between the two
proteomics platforms before this merge was allowed. Its conclusion governs how the two
tables may be used: they are stored as **separate layers**, never concatenated into one
matrix, so a protein can never look "high" merely because it was measured on the more
sensitive instrument.

This cell quantifies the reach the second platform adds, on that basis.

In [ ]:
ccle_ids   = set(explode_ids(con.execute("SELECT model_id FROM proteomics").df()["model_id"]))
procan_ids = new_layer_ids["procan_proteomics"]

for label, s in [("CCLE/Gygi only", ccle_ids - procan_ids),
                 ("ProCan only",    procan_ids - ccle_ids),
                 ("both platforms", ccle_ids & procan_ids),
                 ("either",         ccle_ids | procan_ids)]:
    print(f"{label:16s} {len(s):5d} ({len(s) / len(all_ids) * 100:5.1f}% of hub)")

print(f"\nlines gained: {len(procan_ids - ccle_ids):,}  "
      f"({len(ccle_ids):,} -> {len(ccle_ids | procan_ids):,}, "
      f"{(len(ccle_ids | procan_ids) / max(len(ccle_ids), 1) - 1) * 100:.0f}% increase)")
print(f"overlap for cross-platform comparison: {len(ccle_ids & procan_ids):,} lines")

## 7. Gene dimension enrichment

Stage 0's `gene` table holds ENSG, names, HUGO symbols and UniProt ids. This section
adds what only the newer references know: authoritative HGNC status, and cancer role
from the Cancer Gene Census.

### 7.1 Was the Stage 0 protein-coding inference sound?

Stage 0 §3.6 built its protein-coding whitelist by unioning `mutations` (coding **and
mutated**, so it under-counts) with `hpa_rna`, and stated plainly that a real biotype
table would be the rigorous source. HGNC is that table.

This is the check that was deferred. It is worth doing honestly in both directions —
a whitelist that is too generous quietly admits non-coding genes into scoring, and one
that is too strict silently drops real ones.

In [ ]:
hgnc_coding = set(hgnc["gene_id"].dropna())
inferred    = set(gene["gene_id"])

both_sets = inferred & hgnc_coding
print(f"Stage 0 inferred whitelist : {len(inferred):,} genes")
print(f"HGNC protein-coding (ENSG) : {len(hgnc_coding):,} genes")
print(f"  agreed by both           : {len(both_sets):,}")
print(f"  inferred only (HGNC says not protein-coding, or no ENSG mapped): "
      f"{len(inferred - hgnc_coding):,}")
print(f"  HGNC only (real coding genes the inference missed): "
      f"{len(hgnc_coding - inferred):,}")
print(f"\nprecision vs HGNC : {len(both_sets) / len(inferred) * 100:5.1f}%")
print(f"recall    vs HGNC : {len(both_sets) / len(hgnc_coding) * 100:5.1f}%")

### 7.2 Build `gene_enriched`

Two joins, both left, `gene` always on the left so no row is created or lost:

- **HGNC on `gene_id`** — direct ENSG match, the reliable axis
- **CGC on symbol** — CGC has no ENSG, so it must go through a symbol. The join uses
  `hgnc_symbols` (current + alias + previous) rather than a bare current-symbol match,
  because COSMIC still uses names HGNC has retired

`is_protein_coding` is HGNC's verdict, kept as its own column beside the Stage 0
inference rather than replacing it — Stage 2 can then choose which definition to
filter on, and the choice stays visible in the data.

In [ ]:
gene_enriched = gene.copy()

# --- HGNC, on gene_id ---------------------------------------------------
hgnc_join = hgnc.dropna(subset=["gene_id"]).drop_duplicates("gene_id")[
    ["gene_id", "hgnc_id", "hgnc_symbol", "name", "locus_group", "locus_type", "status"]]
hgnc_join = hgnc_join.rename(columns={"name": "gene_full_name", "status": "hgnc_status"})

before = len(gene_enriched)
gene_enriched = gene_enriched.merge(hgnc_join, on="gene_id", how="left")
assert len(gene_enriched) == before, f"HGNC join fanned out: {before} -> {len(gene_enriched)}"
gene_enriched["is_protein_coding"] = gene_enriched["locus_group"].eq("protein-coding gene")

# --- CGC, through the alias-aware symbol table --------------------------
# Alias expansion cuts both ways: it recovers genes COSMIC still calls by a
# retired name, but an alias shared by two genes would hand a cancer role to the
# wrong one. So the match is taken at the STRONGEST symbol tier that fires for
# each census gene, and anything still ambiguous at that tier is dropped rather
# than picked from.
SYM_RANK = {"current": 0, "previous": 1, "alias": 2}

cgc_role = cosmic_cgc[["cgc_symbol", "gene_role", "tier"]].copy()
cgc_role["sym"] = cgc_role["cgc_symbol"].astype("string").str.strip().str.lower()

hits = (cgc_role.merge(hgnc_symbols, on="sym", how="inner").dropna(subset=["gene_id"]))
hits["_rank"] = hits["symbol_kind"].map(SYM_RANK)
best = hits.groupby("cgc_symbol")["_rank"].transform("min")
hits = hits[hits["_rank"] == best].drop_duplicates(["cgc_symbol", "gene_id"])

n_cgc = cgc_role["cgc_symbol"].nunique()
per_sym = hits.groupby("cgc_symbol")["gene_id"].nunique()
ambiguous_cgc = set(per_sym[per_sym > 1].index)
hits = hits[~hits["cgc_symbol"].isin(ambiguous_cgc)]

print(f"CGC symbols            : {n_cgc:,}")
print(f"  resolved to one ENSG : {hits.cgc_symbol.nunique():,}")
print(f"  ambiguous (>1 ENSG)  : {len(ambiguous_cgc):,}  (dropped, not guessed)")
print(f"  no HGNC symbol route : {n_cgc - per_sym.index.nunique():,}")
print("  matched at symbol tier:", hits.symbol_kind.value_counts().to_dict())

# one gene, one role: 'both' wins over a single-sided call, tier 1 over tier 2
_rank = {"both": 3, "tsg": 2, "oncogene": 2, "unknown": 0}
cgc_by_gene = hits.assign(_r=hits["gene_role"].map(_rank).fillna(0))
cgc_by_gene = (cgc_by_gene.sort_values(["gene_id", "_r", "tier"], ascending=[True, False, True])
                          .drop_duplicates("gene_id")[["gene_id", "gene_role", "tier", "cgc_symbol"]]
                          .rename(columns={"tier": "cgc_tier"}))

before = len(gene_enriched)
gene_enriched = gene_enriched.merge(cgc_by_gene, on="gene_id", how="left")
assert len(gene_enriched) == before, f"CGC join fanned out: {before} -> {len(gene_enriched)}"
gene_enriched["gene_role"] = gene_enriched["gene_role"].fillna("unknown")

print(f"\ngene_enriched: {gene_enriched.shape}  | gene_id unique: {gene_enriched.gene_id.is_unique}")
print(f"  with hgnc_id          : {gene_enriched.hgnc_id.notna().sum():,}")
print(f"  is_protein_coding     : {int(gene_enriched.is_protein_coding.sum()):,}")
print(f"  with a CGC role       : {(gene_enriched.gene_role != 'unknown').sum():,}")
print("\ngene_role:")
print(gene_enriched.gene_role.value_counts().to_string())

### 7.3 Resolve `cosmic_cna` onto the gene axis

The CNA table is symbol-keyed. Now that the alias table exists, its symbols can be
resolved to `gene_id` — which is what lets copy number be joined to expression,
mutation and everything else on one gene key.

Resolution happens on the distinct symbol list, not the 1.4 M rows, and symbols that
resolve to several ENSGs are **dropped from the gene mapping rather than assigned
arbitrarily** — the row survives with its symbol intact and `gene_id` null. Guessing
here would put copy number on the wrong gene, which is worse than leaving it unjoined.

In [ ]:
sym_to_gene = (hgnc_symbols.groupby("sym")["gene_id"].agg(lambda s: sorted(set(s))))
unique_sym  = sym_to_gene[sym_to_gene.map(len) == 1].map(lambda s: s[0])

cna_syms = pd.DataFrame({"cosmic_gene_symbol": cosmic_cna["cosmic_gene_symbol"].unique()})
cna_syms["gene_id"] = cna_syms["cosmic_gene_symbol"].map(unique_sym)
cna_syms["n_candidates"] = cna_syms["cosmic_gene_symbol"].map(
    sym_to_gene.map(len)).fillna(0).astype(int)

n_amb = int((cna_syms.n_candidates > 1).sum())
print(f"distinct CNA symbols     : {len(cna_syms):,}")
print(f"  resolved to one ENSG   : {cna_syms.gene_id.notna().sum():,}")
print(f"  ambiguous (>1 ENSG)    : {n_amb:,}  (left unresolved, not guessed)")
print(f"  no HGNC symbol at all  : {int((cna_syms.n_candidates == 0).sum()):,}")

cosmic_cna = cosmic_cna.merge(cna_syms[["cosmic_gene_symbol", "gene_id"]],
                              on="cosmic_gene_symbol", how="left")
print(f"\nCNA rows on the gene axis: {cosmic_cna.gene_id.notna().sum():,} "
      f"/ {len(cosmic_cna):,} ({cosmic_cna.gene_id.notna().mean() * 100:.1f}%)")
print(f"joinable rows (gene AND line resolved): "
      f"{int((cosmic_cna.gene_id.notna() & cosmic_cna.model_id.notna()).sum()):,}")

## 8. Persist

Written back into the same `celllineselector.db`. Stage 0's tables are never
overwritten — everything here is either a new source table or a `*_enriched` variant
sitting beside the original, so the two waves stay separable and Stage 0 can be re-run
without this notebook having to be re-run first.

`CREATE OR REPLACE` makes the cell safe to rerun.

In [ ]:
to_load = {
    "gdsc_models":                 gdsc_models,
    "procan_proteomics":           procan_proteomics,
    "procan_protein_map":          procan_protein_map,
    "cosmic_cna":                  cosmic_cna,
    "cosmic_cgc":                  cosmic_cgc,
    "hgnc":                        hgnc,
    "hgnc_symbols":                hgnc_symbols,
    "cell_line_connection_enriched": enriched_hub,
    "coverage_matrix_enriched":    coverage_enriched,
    "gene_enriched":               gene_enriched,
}

for name, df in to_load.items():
    con.register("_tmp", df)
    con.execute(f'CREATE OR REPLACE TABLE "{name}" AS SELECT * FROM _tmp')
    con.unregister("_tmp")
    print(f"  {name:32s} {str(df.shape):>16s}")

print(f"\nwrote {len(to_load)} tables to {DB_PATH}")

### 8.1 Parquet exports for the Stage 2-7 scripts

The Stage 2-7 scripts read parquet by path, not SQL. These exports keep that contract,
mirroring Stage 0 §9.

`harmonised_enriched.parquet` is deliberately **not** written here — that name belongs
to Stage 1's output (`01_lookup_metadata_join.ipynb`). Writing it would overwrite a
downstream artefact with an upstream one.

In [ ]:
EXPORTS = {
    "cell_line_connection_enriched.parquet": enriched_hub,
    "coverage_matrix_enriched.parquet":      coverage_enriched,
    "gene_enriched.parquet":                 gene_enriched,
    "gdsc_models.parquet":                   gdsc_models,
    "cosmic_cna.parquet":                    cosmic_cna,
    "procan_proteomics.parquet":             procan_proteomics,
}

for fname, df in EXPORTS.items():
    path = os.path.join(OUT_DIR, fname)
    df.to_parquet(path, index=False)
    print(f"  {fname:42s} {str(df.shape):>16s} -> {os.path.getsize(path) / 1e6:7.1f} MB")

## 9. Verify the load

Reopens read-only and confirms every table landed with the expected shape. This is the
only place the connection is closed.

In [ ]:
con.close()
con = duckdb.connect(DB_PATH, read_only=True)

rows = []
for t in con.execute("SHOW TABLES").df()["name"]:
    cs = [r[0] for r in con.execute(f'DESCRIBE "{t}"').fetchall()]
    rows.append({
        "table": t,
        "wave": "enriched" if t in to_load else "core",
        "rows": con.execute(f'SELECT count(*) FROM "{t}"').fetchone()[0],
        "cols": len(cs),
        "distinct_model_id": (con.execute(f'SELECT count(DISTINCT model_id) FROM "{t}"').fetchone()[0]
                              if "model_id" in cs else None),
    })
print(pd.DataFrame(rows).sort_values(["wave", "rows"], ascending=[True, False])
        .to_string(index=False))

print("\n--- coverage: core vs enriched ---")
print(con.execute("""
    SELECT c.n_modalities AS core_of_9, e.n_modalities AS enriched_of_11, count(*) AS n_lines
    FROM coverage_matrix_enriched e
    LEFT JOIN coverage_matrix c USING (model_id)
    GROUP BY 1, 2 ORDER BY 2 DESC, 1 DESC LIMIT 15
""").df().to_string(index=False))

con.close()
print("\nconnection closed")